In [ ]:
import pandas as pd
df = pd.read_csv("/content/drive/MyDrive/industrial-ai-project/MetroPT3(AirCompressor).csv")
df['timestamp'] = pd.to_datetime(df['timestamp'])
df = df.drop(columns=['Unnamed: 0'])

In [ ]:
df['time_diff'] = df['timestamp'].diff()

# Define what counts as a "big gap" that breaks continuity
# We'll use 1 minute as the threshold (6x the normal ~10s interval)
gap_threshold = pd.Timedelta(minutes=1)

# Create a new column: True whenever a big gap starts a new segment
df['new_segment'] = df['time_diff'] > gap_threshold

# Count how many segments we'll end up with
print("Number of continuity segments:", df['new_segment'].sum() + 1)

Number of continuity segments: 332


In [ ]:
# Aggregate from 10-second readings to 1-minute readings
# This reduces noise and makes anomaly patterns more distinguishable,
# following the approach used in reference projects on this same dataset

# Set timestamp as index (required for resample)
df = df.set_index('timestamp')

# Resample to 1-minute intervals, taking the mean of all readings within each minute
df_agg = df.resample('1min').mean()

# Drop any minutes that had no readings at all (empty gaps)
df_agg = df_agg.dropna(how='all')

# Reset index so timestamp becomes a column again
df_agg = df_agg.reset_index()

print("Rows before aggregation:", len(df))
print("Rows after aggregation:", len(df_agg))

# Replace df with the aggregated version for all following steps
df = df_agg

Rows before aggregation: 1516948
Rows after aggregation: 252720


In [ ]:
# Convert the True/False "new segment" flags into a running segment ID
df['segment_id'] = df['new_segment'].cumsum()

# Check: how many rows does each segment have, on average?
print(df['segment_id'].value_counts().describe())

count     332.000000
mean      761.204819
std       958.745420
min         5.000000
25%        26.000000
50%       387.000000
75%      1195.250000
max      6047.000000
Name: count, dtype: float64


In [ ]:
# Find the segment sizes for every segment
segment_sizes = df['segment_id'].value_counts().sort_values()

# Show the 5 smallest segments
print(segment_sizes.head(5))

segment_id
10.516667     5
115.819048    5
109.269048    5
110.669048    5
35.333333     5
Name: count, dtype: int64


In [ ]:
smallest_segment = df[df['segment_id'] == 47]
print(smallest_segment[['timestamp', 'TP2', 'COMP']])

Empty DataFrame
Columns: [timestamp, TP2, COMP]
Index: []


In [ ]:
# Create rolling window features for TP2 (window = 5 readings ≈ 1 minute)
df['TP2_rolling_mean'] = df['TP2'].rolling(window=5).mean()
df['TP2_rolling_max'] = df['TP2'].rolling(window=5).max()
df['TP2_rolling_std'] = df['TP2'].rolling(window=5).std()

# Show a sample of the new columns
df[['timestamp', 'TP2', 'TP2_rolling_mean', 'TP2_rolling_max', 'TP2_rolling_std']].head(15)

,timestamp,TP2,TP2_rolling_mean,TP2_rolling_max,TP2_rolling_std
0,2020-02-01 00:00:00,-0.012286,NaN,NaN,NaN
1,2020-02-01 00:01:00,-0.013000,NaN,NaN,NaN
2,2020-02-01 00:02:00,-0.012667,NaN,NaN,NaN
3,2020-02-01 00:03:00,-0.012333,NaN,NaN,NaN
4,2020-02-01 00:04:00,-0.013000,-0.012657,-0.012286,0.000346
5,2020-02-01 00:05:00,-0.013667,-0.012933,-0.012333,0.000494
6,2020-02-01 00:06:00,-0.013333,-0.013000,-0.012333,0.000527
7,2020-02-01 00:07:00,-0.013000,-0.013067,-0.012333,0.000494
8,2020-02-01 00:08:00,-0.013667,-0.013333,-0.013000,0.000333
9,2020-02-01 00:09:00,-0.013333,-0.013400,-0.013000,0.000279


## Correction: Rolling features must respect continuity segments

The rolling features below were initially computed across the whole
file, ignoring timestamp gaps. This caused misleading values near large
gaps (e.g., mixing readings from before and after a multi-hour gap).
Recomputing them grouped by `segment_id` instead.

In [ ]:
# Corrected: apply rolling statistics WITHIN each continuity segment,
# so we never mix readings across a timestamp gap

key_sensors = ['TP2', 'H1', 'Oil_temperature', 'Motor_current']

for sensor in key_sensors:
    grouped = df.groupby('segment_id')[sensor]
    df[f'{sensor}_rolling_mean'] = grouped.transform(lambda x: x.rolling(window=5).mean())
    df[f'{sensor}_rolling_max'] = grouped.transform(lambda x: x.rolling(window=5).max())
    df[f'{sensor}_rolling_std'] = grouped.transform(lambda x: x.rolling(window=5).std())

# Check the new columns were created
print(df.columns.tolist())

['timestamp', 'TP2', 'TP3', 'H1', 'DV_pressure', 'Reservoirs', 'Oil_temperature', 'Motor_current', 'COMP', 'DV_eletric', 'Towers', 'MPG', 'LPS', 'Pressure_switch', 'Oil_level', 'Caudal_impulses', 'time_diff', 'new_segment', 'segment_id', 'TP2_rolling_mean', 'TP2_rolling_max', 'TP2_rolling_std', 'H1_rolling_mean', 'H1_rolling_max', 'H1_rolling_std', 'Oil_temperature_rolling_mean', 'Oil_temperature_rolling_max', 'Oil_temperature_rolling_std', 'Motor_current_rolling_mean', 'Motor_current_rolling_max', 'Motor_current_rolling_std']


In [ ]:
# Check how many rows have NaN before removing them (sanity check)
print("Rows before dropping:", len(df))
print("Rows with any NaN in rolling features:", df.isnull().any(axis=1).sum())

# Remove rows with NaN (start of each continuity segment, due to rolling window)
df = df.dropna().reset_index(drop=True)

print("Rows after dropping:", len(df))

Rows before dropping: 252720
Rows with any NaN in rolling features: 1328
Rows after dropping: 251392


In [ ]:
# Check if there are any large gaps DURING failure #4 itself (not just before it)
failure_4_start = pd.to_datetime("2020-07-15 14:30:00")
failure_4_end = pd.to_datetime("2020-07-15 19:00:00")

gaps_during_failure_4 = df[
    (df['timestamp'] >= failure_4_start) &
    (df['timestamp'] <= failure_4_end) &
    (df['time_diff'] > pd.Timedelta(minutes=1))
]

print("Gaps found during Failure #4 itself:", len(gaps_during_failure_4))

Gaps found during Failure #4 itself: 0


## Decision: Failure #4 is included in the is_failure target

Earlier, we excluded Failure #4 (2020-07-15) from time-sensitive
"pre-failure" analysis due to a 14h11m gap ending ~3h before its onset.

However, since the project now targets Failure Detection (is the
compressor currently failing?) rather than Failure Prediction, and we
confirmed zero data gaps during the failure period itself (14:30–19:00),
Failure #4 can be safely included in the is_failure target.

In [ ]:
# Define all 4 known failure periods (start, end) from official report
failure_periods = [
    (pd.to_datetime("2020-04-18 00:00:00"), pd.to_datetime("2020-04-18 23:59:59")),
    (pd.to_datetime("2020-05-29 23:30:00"), pd.to_datetime("2020-05-30 06:00:00")),
    (pd.to_datetime("2020-06-05 10:00:00"), pd.to_datetime("2020-06-07 14:30:00")),
    (pd.to_datetime("2020-07-15 14:30:00"), pd.to_datetime("2020-07-15 19:00:00")),
]

# Start with everything labeled as "not failure" (0)
df['is_failure'] = 0

# Mark rows that fall inside any failure period as 1
for start, end in failure_periods:
    df.loc[(df['timestamp'] >= start) & (df['timestamp'] <= end), 'is_failure'] = 1

# Check how many rows are labeled as failure vs normal
print(df['is_failure'].value_counts())

is_failure
0    246448
1      4944
Name: count, dtype: int64


In [ ]:
# Chronological split: train on everything up to end of Failure #3,
# test on the rest (includes Failure #4 fully, plus a longer evaluation period)
split_date = pd.to_datetime("2020-06-08 00:00:00")

train_df = df[df['timestamp'] < split_date]
test_df = df[df['timestamp'] >= split_date]

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)
print("Train failure ratio:", train_df['is_failure'].mean())
print("Test failure ratio:", test_df['is_failure'].mean())

Train shape: (150265, 32)
Test shape: (101127, 32)
Train failure ratio: 0.031131667387615213
Test failure ratio: 0.0026303558891295105


## Note: Final Train/Test Split

Train (Feb - June 7): 150,071 rows, 3.12% failure rate — includes
Failures #1, #2, #3 fully.

Test (June 8 onward): 100,989 rows, 0.26% failure rate — includes
Failure #4 fully, plus a long stretch of normal operation (mid-June
through September).

Note: after aggregating from 10-second to 1-minute intervals, row counts
dropped by ~6x as expected, while failure ratios stayed consistent with
the pre-aggregation split (3.12% train, ~0.26% test). This confirms the
aggregation preserved the same temporal structure.

In [ ]:
import os
os.makedirs("/content/drive/MyDrive/industrial-ai-project/data/processed", exist_ok=True)

In [ ]:
# Save the processed dataframe so later notebooks (like classification)
# don't need to repeat all the cleaning and feature engineering steps
output_path = "/content/drive/MyDrive/industrial-ai-project/data/processed/metropt3_features.csv"
df.to_csv(output_path, index=False)

print("Saved:", df.shape)

Saved: (251392, 32)
